# Chunking cho RAG - Bộ code Colab đầy đủ

Notebook này cài đặt **từ đầu (from scratch)** và kèm bản **LangChain** (khi có) cho hầu hết các kỹ thuật chunking.

| Nhóm | Kỹ thuật |
|---|---|
| A. Cơ bản | Fixed-size (ký tự), Token-based (tiktoken + HF tokenizer), Word-based, Sentence-based, Paragraph-based, Recursive, Sentence-window |
| B. Structure-based | Markdown header, Element/Block-aware (bảng, code, list), HTML, Code (Python AST), JSON, PDF/DOCX loader |
| C. Semantic | Embedding breakpoint (percentile / stddev / IQR / gradient), Lexical TextTiling, Clustering liền kề, LangChain SemanticChunker |
| D. Nâng cao | Parent-Child (small-to-big), Late chunking, LLM/Agentic chunking, Propositional, Contextual chunking |
| E. Đánh giá | Thống kê chunk, chất lượng ranh giới, Hit@k / MRR trên bộ câu hỏi mẫu |

**Cách dùng:** chạy lần lượt từ trên xuống (Runtime -> Run all). Phần D có các cell dùng LLM cần API key, nếu không có key thì cell tự bỏ qua hoặc dùng LLM giả lập.

## 0. Cài đặt & cấu hình

In [ ]:
# (numpy, pandas, scikit-learn, matplotlib, torch đã có sẵn trên Colab)
!pip install -q sentence-transformers tiktoken beautifulsoup4 python-docx langchain-text-splitters langchain-experimental langchain-huggingface

In [ ]:
import re, json, ast, math, os, warnings
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Callable, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Mô hình embedding đa ngôn ngữ (có tiếng Việt), nhẹ, chạy tốt trên Colab CPU.
# Có thể đổi sang: "intfloat/multilingual-e5-base", "BAAI/bge-m3", "AITeamVN/Vietnamese_Embedding"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 256          # đủ cho cửa sổ 3 câu
hf_tokenizer = embedder.tokenizer      # tokenizer "fast" -> có offset_mapping

def embed(texts: List[str]) -> np.ndarray:
    """Trả về ma trận embedding đã chuẩn hoá L2 (dot product = cosine)."""
    return embedder.encode(list(texts), normalize_embeddings=True,
                           batch_size=32, show_progress_bar=False)

# Đếm token (ước lượng theo tiktoken, fallback sang đếm từ nếu không tải được)
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(t: str) -> int:
        return len(_enc.encode(t))
except Exception:
    _enc = None
    def count_tokens(t: str) -> int:
        return max(1, int(len(t.split()) * 1.3))

### Cấu trúc dữ liệu & tiện ích dùng chung

In [ ]:
@dataclass
class Chunk:
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    def __len__(self):
        return len(self.text)

def to_chunks(texts: List[str], method: str, **meta) -> List[Chunk]:
    return [Chunk(t, {"method": method, "idx": i, **meta}) for i, t in enumerate(texts)]

def texts_of(chunks) -> List[str]:
    return [c.text if isinstance(c, Chunk) else c for c in chunks]

def show(chunks, n=4, width=260):
    """In nhanh vài chunk đầu để xem."""
    ts = texts_of(chunks)
    print(f"-> {len(ts)} chunks | trung bình {np.mean([len(t) for t in ts]):.0f} ký tự")
    for i, t in enumerate(ts[:n]):
        t1 = t.replace("\n", " ⏎ ")
        print(f"  [{i}] ({len(t)} ký tự) {t1[:width]}{'...' if len(t1) > width else ''}")
    if len(ts) > n:
        print(f"  ... còn {len(ts) - n} chunk nữa")

# --- Tách câu (hỗ trợ tiếng Việt + Anh, tránh tách sai ở viết tắt) ---
_ABBR = re.compile(r"(?:\b(?:TS|PGS|GS|ThS|TP|Mr|Mrs|Dr|vs|etc|e\.g|i\.e)|\bv\.v)\.$")

def split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text.strip())
    parts = re.split(r"(?<=[.!?…])\s+", text)
    out: List[str] = []
    for p in parts:
        if not p:
            continue
        if out and (p[:1].islower() or _ABBR.search(out[-1])):
            out[-1] += " " + p
        else:
            out.append(p)
    return out

def md_to_plain(md: str) -> str:
    """Markdown -> văn bản thuần: bỏ code, đổi bảng thành câu, bỏ ký hiệu #, -, **.
    Tiêu đề và mục list được thêm dấu chấm để trở thành 'câu' riêng khi tách câu."""
    in_code, out = False, []
    for ln in md.splitlines():
        if ln.strip().startswith(("```", "~~~")):
            in_code = not in_code
            if in_code:                                      # "Ví dụ ... :" đứng trước code -> kết thúc bằng "."
                k = len(out) - 1
                while k >= 0 and not out[k].strip():
                    k -= 1
                if k >= 0 and out[k].rstrip().endswith(":"):
                    out[k] = out[k].rstrip()[:-1] + "."
            continue
        if in_code:
            continue
        if re.match(r"^\s*\|?\s*:?-{3,}", ln):          # dòng phân cách bảng
            continue
        if ln.strip().startswith("|"):                  # dòng bảng -> "a; b; c."
            cells = [c.strip() for c in ln.strip().strip("|").split("|")]
            ln = "; ".join(cells) + "."
        is_head = bool(re.match(r"^\s*#{1,6}\s+", ln))
        ln = re.sub(r"^\s*#{1,6}\s+", "", ln)           # header
        ln = re.sub(r"^\s*[-*+]\s+", "", ln)            # bullet
        ln = re.sub(r"[*_`]{1,3}", "", ln)
        if is_head and ln.strip() and ln.strip()[-1] not in ".!?…:;":
            ln = ln.rstrip() + "."
        out.append(ln)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()

### Dữ liệu mẫu (Markdown): hai chủ đề khác nhau để thấy rõ Semantic chunking cắt đúng chỗ

In [ ]:
DOC_MD = r'''# Sổ tay tổng hợp: RAG và vùng đất Cần Thơ

## Phần 1 - Retrieval-Augmented Generation (RAG)

### 1.1 RAG là gì?

RAG (Retrieval-Augmented Generation) là kỹ thuật kết hợp truy xuất thông tin với mô hình ngôn ngữ lớn. Thay vì chỉ dựa vào kiến thức đã học khi huấn luyện, mô hình sẽ được cung cấp thêm các đoạn tài liệu liên quan tại thời điểm trả lời. Nhờ vậy câu trả lời chính xác hơn, cập nhật hơn và giảm hiện tượng bịa đặt thông tin. Hệ thống RAG thường gồm ba bước: lập chỉ mục tài liệu, truy xuất các đoạn phù hợp và sinh câu trả lời.

### 1.2 Vì sao cần chunking?

Tài liệu thực tế thường dài hàng chục nghìn từ, trong khi mô hình embedding chỉ xử lý được một số lượng token giới hạn. Vì vậy ta phải chia tài liệu thành các đoạn nhỏ gọi là chunk. Chunk quá lớn sẽ chứa nhiều chủ đề, làm vector embedding bị "loãng" và truy xuất kém chính xác. Chunk quá nhỏ lại mất ngữ cảnh, khiến mô hình không đủ thông tin để trả lời. Việc chọn kích thước và chiến lược chia phù hợp là một trong những yếu tố ảnh hưởng nhiều nhất đến chất lượng của RAG.

### 1.3 Các chiến lược chunking phổ biến

- Fixed-size: cắt theo số ký tự hoặc token cố định, đơn giản và nhanh nhưng dễ cắt giữa câu.
- Recursive: thử tách theo đoạn, rồi câu, rồi từ cho đến khi đủ nhỏ.
- Structure-based: dựa vào tiêu đề, bảng, code block của tài liệu.
- Semantic: dùng embedding để phát hiện nơi chủ đề thay đổi.
- LLM-based: nhờ mô hình ngôn ngữ quyết định ranh giới chunk.

| Chiến lược | Ưu điểm | Nhược điểm |
|---|---|---|
| Fixed-size | Nhanh, dễ cài đặt | Dễ cắt giữa ý |
| Recursive | Cân bằng, phổ biến nhất | Không hiểu ngữ nghĩa |
| Semantic | Chunk mạch lạc theo chủ đề | Tốn chi phí tính embedding |
| LLM-based | Chất lượng cao nhất | Chậm và đắt |

Ví dụ cài đặt chunking cố định bằng Python:

```python
def fixed_chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]
```

### 1.4 Embedding và cơ sở dữ liệu vector

Sau khi chia nhỏ, mỗi chunk được chuyển thành một vector bằng mô hình embedding. Các vector được lưu trong cơ sở dữ liệu vector như FAISS, Chroma hoặc Qdrant. Khi người dùng đặt câu hỏi, câu hỏi cũng được chuyển thành vector và hệ thống tìm ra các chunk gần nhất theo độ tương đồng cosine. Một mẹo hữu ích là dùng chunk overlap khoảng 10 đến 20 phần trăm để không mất thông tin ở ranh giới.

## Phần 2 - Giới thiệu thành phố Cần Thơ

### 2.1 Vị trí và đặc điểm

Cần Thơ là thành phố lớn nhất vùng Đồng bằng sông Cửu Long, nằm bên bờ phải sông Hậu. Thành phố được mệnh danh là "Tây Đô" nhờ vị trí trung tâm của vùng miền Tây Nam Bộ. Khí hậu nhiệt đới gió mùa với hai mùa mưa và nắng rõ rệt. Hệ thống sông ngòi, kênh rạch chằng chịt tạo nên nét văn hóa sông nước đặc trưng.

### 2.2 Địa điểm tham quan

Chợ nổi Cái Răng là điểm đến nổi tiếng nhất, họp từ sáng sớm với hàng trăm ghe thuyền mua bán nông sản. Bến Ninh Kiều là nơi người dân và du khách tản bộ, ngắm sông vào buổi tối. Ngoài ra, nhà cổ Bình Thủy và vườn cò Bằng Lăng cũng là những địa điểm đáng ghé thăm. Du khách nên đến chợ nổi trước 7 giờ sáng để thấy không khí nhộn nhịp nhất.

### 2.3 Ẩm thực

Cần Thơ nổi tiếng với bánh xèo, lẩu mắm, bún cá và bánh cống. Lẩu mắm được nấu từ mắm cá linh hoặc mắm cá sặc, ăn kèm nhiều loại rau đồng. Các món ăn ở đây thường có vị ngọt đậm đà, đặc trưng của ẩm thực miền Tây.
'''

DOC_PLAIN = md_to_plain(DOC_MD)
print(f"Markdown: {len(DOC_MD)} ký tự | Plain: {len(DOC_PLAIN)} ký tự | ~{count_tokens(DOC_PLAIN)} token")
print(len(split_sentences(DOC_PLAIN)), "câu")

---
# A. Chunking cơ bản
## A1. Fixed-size (theo ký tự) + overlap
Nhanh, dễ, dự đoán được kích thước. Nhược điểm: không biết ngữ nghĩa nên dễ cắt giữa câu. Tùy chọn `respect_words=True` giúp không cắt giữa từ.

In [ ]:
def _rfind_ws(t: str, lo: int, hi: int) -> int:
    return max(t.rfind(" ", lo, hi), t.rfind("\n", lo, hi))

def _find_ws(t: str, lo: int, hi: int) -> int:
    cands = [i for i in (t.find(" ", lo, hi), t.find("\n", lo, hi)) if i != -1]
    return min(cands) if cands else -1

def fixed_size_chunk(text: str, chunk_size: int = 500, overlap: int = 50,
                     respect_words: bool = True) -> List[str]:
    assert 0 <= overlap < chunk_size, "overlap phải nhỏ hơn chunk_size"
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + chunk_size, n)
        if respect_words and end < n:                       # lùi về khoảng trắng gần nhất
            sp = _rfind_ws(text, start + chunk_size // 2, end)
            if sp != -1:
                end = sp
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        nxt = end - overlap
        start = nxt if nxt > start else end
        if respect_words and 0 < start < n and not text[start - 1].isspace():
            sp = _find_ws(text, start, end)                # tiến tới ranh giới từ kế tiếp
            if sp != -1:
                start = sp + 1
    return chunks

c_fixed = fixed_size_chunk(DOC_PLAIN, chunk_size=400, overlap=60)
show(c_fixed)

## A2. Token-based (theo token) + overlap
Quan trọng khi cần đảm bảo chunk **không vượt giới hạn token** của mô hình embedding/LLM.
- Bản 1: `tiktoken` (cl100k_base, dùng cho OpenAI).
- Bản 2: tokenizer của chính mô hình embedding (HuggingFace) - cắt theo `offset_mapping` nên **không bao giờ cắt hỏng ký tự tiếng Việt**.

In [ ]:
def token_chunk_tiktoken(text: str, chunk_tokens: int = 128, overlap_tokens: int = 16) -> List[str]:
    if _enc is None:
        raise RuntimeError("tiktoken không khả dụng, hãy dùng token_chunk_hf")
    ids = _enc.encode(text)
    step = chunk_tokens - overlap_tokens
    out = []
    for i in range(0, len(ids), step):
        out.append(_enc.decode(ids[i:i + chunk_tokens]))
        if i + chunk_tokens >= len(ids):
            break
    return out   # lưu ý: tiktoken byte-level có thể làm hỏng ký tự ở mép chunk với tiếng Việt

def token_chunk_hf(text: str, tokenizer=hf_tokenizer, chunk_tokens: int = 128,
                   overlap_tokens: int = 16) -> List[str]:
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True,
                    truncation=False, verbose=False)
    offs = enc["offset_mapping"]
    step = chunk_tokens - overlap_tokens
    out = []
    for i in range(0, len(offs), step):
        j = min(i + chunk_tokens, len(offs)) - 1
        out.append(text[offs[i][0]: offs[j][1]].strip())
        if i + chunk_tokens >= len(offs):
            break
    return out

c_tok_hf = token_chunk_hf(DOC_PLAIN, chunk_tokens=100, overlap_tokens=10)
show(c_tok_hf)
print("Token/chunk (HF):", [len(hf_tokenizer(t, add_special_tokens=False)["input_ids"]) for t in c_tok_hf])
if _enc is not None:
    c_tok_tt = token_chunk_tiktoken(DOC_PLAIN, 100, 10)
    print("tiktoken:", len(c_tok_tt), "chunks")

## A3. Word-based

In [ ]:
def word_chunk(text: str, words_per_chunk: int = 80, overlap_words: int = 10) -> List[str]:
    words = text.split()
    step = words_per_chunk - overlap_words
    out = []
    for i in range(0, len(words), step):
        out.append(" ".join(words[i:i + words_per_chunk]))
        if i + words_per_chunk >= len(words):
            break
    return out

show(word_chunk(DOC_PLAIN, 60, 8))

## A4. Sentence-based (gom câu, overlap theo câu)
Không bao giờ cắt giữa câu. Gom câu đến khi đạt `max_chars`, giữ lại `overlap_sentences` câu cuối cho chunk kế tiếp.

In [ ]:
def sentence_chunk(text: str, max_chars: int = 450, overlap_sentences: int = 1) -> List[str]:
    sents = split_sentences(text)
    chunks, cur, cur_len = [], [], 0
    for s in sents:
        if cur and cur_len + len(s) + 1 > max_chars:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sentences:] if overlap_sentences else []
            cur_len = sum(len(x) + 1 for x in cur)
        cur.append(s)
        cur_len += len(s) + 1
    if cur:
        chunks.append(" ".join(cur))
    return chunks

c_sent = sentence_chunk(DOC_PLAIN, 450, 1)
show(c_sent)

## A5. Paragraph-based
Tách theo đoạn văn (`\n\n`), gộp đoạn ngắn lại cho đủ `min_chars`, không vượt `max_chars` (đoạn quá dài sẽ được tách theo câu).

In [ ]:
def paragraph_chunk(text: str, max_chars: int = 700, min_chars: int = 200) -> List[str]:
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(p) > max_chars:                       # đoạn quá dài -> tách theo câu
            if cur:
                chunks.append(cur); cur = ""
            chunks.extend(sentence_chunk(p, max_chars, 0))
            continue
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur); cur = p
        elif cur and len(cur) >= min_chars:
            chunks.append(cur); cur = p
        else:
            cur = f"{cur}\n\n{p}" if cur else p
    if cur:
        chunks.append(cur)
    return chunks

c_para = paragraph_chunk(DOC_PLAIN, 700, 200)
show(c_para)

## A6. Recursive character splitting
Phổ biến nhất (mặc định của LangChain): thử tách theo dấu phân cách **từ lớn đến nhỏ** (đoạn -> dòng -> câu -> từ -> ký tự), gộp mảnh nhỏ lại đến `chunk_size`.

In [ ]:
DEFAULT_SEPS = ["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""]

def _split_keep(text: str, sep: str) -> List[str]:
    if sep == "":
        return list(text)
    parts = text.split(sep)
    return [p + sep for p in parts[:-1]] + [parts[-1]]   # giữ dấu phân cách ở cuối mảnh

def recursive_chunk(text: str, chunk_size: int = 450, overlap: int = 50,
                    seps: List[str] = DEFAULT_SEPS) -> List[str]:
    def _split(t: str, ss: List[str]) -> List[str]:
        if len(t) <= chunk_size:
            return [t]
        sep = next((s for s in ss if s == "" or s in t), "")
        rest = ss[ss.index(sep) + 1:]
        out = []
        for p in _split_keep(t, sep):
            out.extend([p] if len(p) <= chunk_size else _split(p, rest))
        return out

    atoms = _split(text, list(seps))
    chunks, cur, cur_len = [], [], 0
    for a in atoms:
        if cur and cur_len + len(a) > chunk_size:
            chunks.append("".join(cur).strip())
            keep, l = [], 0                                 # overlap: giữ các mảnh cuối
            for x in reversed(cur):
                if l + len(x) > overlap:
                    break
                keep.insert(0, x); l += len(x)
            while keep and l + len(a) > chunk_size:
                l -= len(keep.pop(0))
            cur, cur_len = keep, l
        cur.append(a); cur_len += len(a)
    if cur:
        chunks.append("".join(cur).strip())
    return [c for c in chunks if c]

c_rec = recursive_chunk(DOC_PLAIN, 450, 50)
show(c_rec)

# --- Bản LangChain (tương đương) ---
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    lc_rec = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50,
                                            separators=["\n\n", "\n", ". ", " ", ""])
    print("\n[LangChain] Recursive:")
    show(lc_rec.split_text(DOC_PLAIN))
except Exception as e:
    print("Bỏ qua LangChain:", e)

## A7. Sentence-window (small-to-big theo câu)
Mỗi **câu** là một chunk để embedding/truy xuất (chính xác), nhưng lưu kèm **cửa sổ ± k câu** làm ngữ cảnh đưa cho LLM.

In [ ]:
def sentence_window_chunks(text: str, window: int = 2) -> List[Chunk]:
    sents = split_sentences(text)
    return [Chunk(s, {"method": "sentence_window", "idx": i,
                      "window_text": " ".join(sents[max(0, i - window): i + window + 1])})
            for i, s in enumerate(sents)]

c_sw = sentence_window_chunks(DOC_PLAIN, 1)
print(f"{len(c_sw)} chunk. Ví dụ:\n  text   : {c_sw[3].text}\n  window : {c_sw[3].metadata['window_text'][:300]}...")

---
# B. Structure-based chunking
Tận dụng cấu trúc có sẵn của tài liệu (tiêu đề, bảng, code, thẻ HTML, cây JSON...) -> chunk giữ nguyên ý nghĩa & có metadata phân cấp.
## B1. Markdown header splitting (giữ đường dẫn tiêu đề trong metadata)

In [ ]:
HEADER_RE = re.compile(r"^(#{1,6})\s+(.*?)\s*#*\s*$")

def markdown_header_chunk(md: str, max_level: int = 3, keep_header: bool = True) -> List[Chunk]:
    sections: List[Chunk] = []
    stack: List[Tuple[int, str]] = []
    buf: List[str] = []
    in_code = False

    def flush():
        body = "\n".join(buf).strip()
        if body and any(l.strip() and not HEADER_RE.match(l) for l in buf):
            meta = {"method": "markdown_header",
                    "header_path": " > ".join(t for _, t in stack),
                    **{f"h{lv}": t for lv, t in stack}}
            sections.append(Chunk(body, meta))

    for line in md.splitlines():
        if line.strip().startswith(("```", "~~~")):
            in_code = not in_code
        m = None if in_code else HEADER_RE.match(line)
        if m and len(m.group(1)) <= max_level:
            flush(); buf.clear()
            lv, title = len(m.group(1)), m.group(2)
            while stack and stack[-1][0] >= lv:
                stack.pop()
            stack.append((lv, title))
            if keep_header:
                buf.append(line)
        else:
            buf.append(line)
    flush()
    return sections

def split_large(chunks: List[Chunk], max_chars: int = 600, overlap: int = 40,
                prefix_path: bool = True) -> List[Chunk]:
    """Chunk quá lớn -> tách nhỏ bằng recursive, gắn lại đường dẫn tiêu đề cho từng mảnh."""
    out = []
    for c in chunks:
        if len(c.text) <= max_chars:
            out.append(c); continue
        for j, piece in enumerate(recursive_chunk(c.text, max_chars, overlap)):
            txt = f"[{c.metadata.get('header_path', '')}]\n{piece}" if prefix_path else piece
            out.append(Chunk(txt, {**c.metadata, "sub_idx": j}))
    return out

c_md = markdown_header_chunk(DOC_MD, max_level=3)
print(f"{len(c_md)} section:")
for c in c_md:
    print(f"  - {c.metadata['header_path']!r:75} {len(c.text):4d} ký tự")
c_md_split = split_large(c_md, max_chars=600)
print("\nSau khi tách section quá dài:", len(c_md_split), "chunks")

# --- Bản LangChain ---
try:
    from langchain_text_splitters import MarkdownHeaderTextSplitter
    mhs = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
                                     strip_headers=False)
    lc_docs = mhs.split_text(DOC_MD)
    print("\n[LangChain] MarkdownHeaderTextSplitter:", len(lc_docs), "docs; metadata mẫu:", lc_docs[2].metadata)
except Exception as e:
    print("Bỏ qua LangChain:", e)

## B2. Element/Block-aware chunking (bảng, code, list được giữ nguyên khối)
Parse Markdown thành các **block** (heading / paragraph / list / table / code). Gom block trong cùng một section đến `max_chars`; **bảng và code không bị cắt ngang** (bảng quá lớn được tách theo hàng và lặp lại dòng tiêu đề).

In [ ]:
LIST_RE = re.compile(r"^\s*([-*+]|\d+[.)])\s+")

def parse_markdown_blocks(md: str) -> List[Dict[str, Any]]:
    lines, i, blocks = md.splitlines(), 0, []
    while i < len(lines):
        ln = lines[i]
        if not ln.strip():
            i += 1; continue
        if ln.strip().startswith(("```", "~~~")):                      # code block
            fence = ln.strip()[:3]; j = i + 1
            while j < len(lines) and not lines[j].strip().startswith(fence):
                j += 1
            blocks.append({"type": "code", "text": "\n".join(lines[i:j + 1])}); i = j + 1
        elif HEADER_RE.match(ln):                                       # heading
            m = HEADER_RE.match(ln)
            blocks.append({"type": "heading", "level": len(m.group(1)), "title": m.group(2), "text": ln}); i += 1
        elif ln.strip().startswith("|"):                                # table
            j = i
            while j < len(lines) and lines[j].strip().startswith("|"):
                j += 1
            blocks.append({"type": "table", "text": "\n".join(lines[i:j])}); i = j
        elif LIST_RE.match(ln):                                         # list
            j = i + 1
            while j < len(lines) and lines[j].strip() and (LIST_RE.match(lines[j]) or lines[j].startswith((" ", "\t"))):
                j += 1
            blocks.append({"type": "list", "text": "\n".join(lines[i:j])}); i = j
        else:                                                           # paragraph
            j = i + 1
            while j < len(lines) and lines[j].strip() and not (
                    HEADER_RE.match(lines[j]) or lines[j].strip().startswith(("```", "~~~", "|")) or LIST_RE.match(lines[j])):
                j += 1
            blocks.append({"type": "paragraph", "text": "\n".join(lines[i:j])}); i = j
    return blocks

def split_table(table_md: str, max_chars: int) -> List[str]:
    rows = table_md.splitlines()
    if len(table_md) <= max_chars or len(rows) <= 3:
        return [table_md]
    head, body = rows[:2], rows[2:]
    out, cur = [], []
    for r in body:
        if cur and len("\n".join(head + cur + [r])) > max_chars:
            out.append("\n".join(head + cur)); cur = []
        cur.append(r)
    if cur:
        out.append("\n".join(head + cur))
    return out

def element_chunk(md: str, max_chars: int = 600, max_heading_level: int = 3) -> List[Chunk]:
    blocks = parse_markdown_blocks(md)
    chunks: List[Chunk] = []
    path: List[Tuple[int, str]] = []
    buf: List[str] = []
    types: List[str] = []
    cur_heading = None

    def flush():
        nonlocal buf, types
        if buf:
            chunks.append(Chunk("\n\n".join(buf), {"method": "element",
                                                   "header_path": " > ".join(t for _, t in path),
                                                   "block_types": sorted(set(types))}))
        buf, types = [], []

    for b in blocks:
        if b["type"] == "heading" and b["level"] <= max_heading_level:
            if not all(t == "heading" for t in types):      # tiêu đề liên tiếp được gộp chung với nội dung bên dưới
                flush()
            while path and path[-1][0] >= b["level"]:
                path.pop()
            path.append((b["level"], b["title"]))
            buf.append(b["text"]); types.append("heading")
            cur_heading = b["text"]
            continue
        pieces = [b["text"]]
        if b["type"] == "table":
            pieces = split_table(b["text"], max_chars)
        elif b["type"] in ("paragraph", "list") and len(b["text"]) > max_chars:
            pieces = sentence_chunk(b["text"], max_chars, 0) if b["type"] == "paragraph" else \
                     recursive_chunk(b["text"], max_chars, 0)
        for p in pieces:
            cur_len = sum(len(x) + 2 for x in buf)
            only_heading = len(buf) == 1 and types == ["heading"]
            if buf and cur_len + len(p) > max_chars and not only_heading:
                flush()
                if cur_heading:
                    buf.append(cur_heading); types.append("heading")   # lặp lại tiêu đề gần nhất cho chunk tiếp theo
            buf.append(p); types.append(b["type"])
    flush()
    return chunks

c_elem = element_chunk(DOC_MD, max_chars=600)
print(f"{len(c_elem)} chunks")
for c in c_elem:
    print(f"  - {c.metadata['header_path'].split(' > ')[-1]!r:45} {c.metadata['block_types']} {len(c.text)} ký tự")

## B3. HTML chunking theo thẻ heading (BeautifulSoup)

In [ ]:
from bs4 import BeautifulSoup

SAMPLE_HTML = """
<html><body>
<h1>Hướng dẫn mua sắm online</h1>
<p>Trang này hướng dẫn các bước đặt hàng và thanh toán an toàn.</p>
<h2>Đặt hàng</h2>
<p>Chọn sản phẩm, thêm vào giỏ hàng rồi nhấn nút Thanh toán.</p>
<ul><li>Kiểm tra địa chỉ giao hàng</li><li>Chọn phương thức vận chuyển</li></ul>
<h2>Thanh toán</h2>
<table>
  <tr><th>Phương thức</th><th>Phí</th></tr>
  <tr><td>Thẻ nội địa</td><td>Miễn phí</td></tr>
  <tr><td>Thẻ quốc tế</td><td>2%</td></tr>
</table>
<h3>Lưu ý</h3>
<p>Không chia sẻ mã OTP cho bất kỳ ai.</p>
</body></html>
"""

def html_header_chunk(html: str, headers=("h1", "h2", "h3")) -> List[Chunk]:
    soup = BeautifulSoup(html, "html.parser")
    root = soup.body or soup
    tags = list(headers) + ["h4", "h5", "h6", "p", "ul", "ol", "table", "pre", "blockquote"]
    stack, buf, chunks = [], [], []

    def flush():
        if buf:
            chunks.append(Chunk("\n".join(buf), {"method": "html_header",
                                                 "header_path": " > ".join(t for _, t in stack)}))
            buf.clear()

    for el in root.find_all(tags):
        if el.find_parent(["ul", "ol", "table", "pre", "blockquote"]):
            continue                                            # đã được xử lý trong phần tử cha
        if el.name in headers:
            flush()
            lv = int(el.name[1])
            while stack and stack[-1][0] >= lv:
                stack.pop()
            stack.append((lv, el.get_text(" ", strip=True)))
            buf.append(el.get_text(" ", strip=True))
        elif el.name == "table":
            rows = [" | ".join(c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"]))
                    for tr in el.find_all("tr")]
            buf.append("\n".join(rows))
        elif el.name in ("ul", "ol"):
            buf.append("\n".join("- " + li.get_text(" ", strip=True) for li in el.find_all("li")))
        else:
            buf.append(el.get_text(" ", strip=True))
    flush()
    return chunks

c_html = html_header_chunk(SAMPLE_HTML)
for c in c_html:
    print(f"[{c.metadata['header_path']}]\n{c.text}\n---")

## B4. Code chunking (Python AST) - mỗi hàm/lớp là một chunk

In [ ]:
SAMPLE_CODE = '''
import math

PI_APPROX = 3.14

def area_circle(r):
    """Diện tích hình tròn."""
    return math.pi * r ** 2

@staticmethod
def clamp(x, lo, hi):
    return max(lo, min(hi, x))

class Shape:
    """Lớp cơ sở cho hình học."""
    def __init__(self, name):
        self.name = name

    def area(self):
        raise NotImplementedError

    def describe(self):
        return f"{self.name} có diện tích {self.area():.2f}"

class Square(Shape):
    def __init__(self, a):
        super().__init__("Hình vuông")
        self.a = a
    def area(self):
        return self.a ** 2
'''

def python_ast_chunk(src: str, max_chars: int = 400) -> List[Chunk]:
    tree = ast.parse(src)
    lines = src.splitlines()
    chunks, module_level = [], []

    def seg(node):
        start = (node.decorator_list[0].lineno if getattr(node, "decorator_list", None) else node.lineno) - 1
        return "\n".join(lines[start:node.end_lineno])

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            chunks.append(Chunk(seg(node), {"method": "python_ast", "type": "function",
                                            "name": node.name, "lineno": node.lineno,
                                            "doc": ast.get_docstring(node)}))
        elif isinstance(node, ast.ClassDef):
            code = seg(node)
            if len(code) <= max_chars:
                chunks.append(Chunk(code, {"method": "python_ast", "type": "class",
                                           "name": node.name, "lineno": node.lineno}))
            else:                                               # lớp lớn -> tách theo method, lặp lại dòng khai báo lớp
                header = lines[node.lineno - 1]
                for sub in node.body:
                    if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                        chunks.append(Chunk(f"{header}\n    ...\n{seg(sub)}",
                                            {"method": "python_ast", "type": "method",
                                             "name": f"{node.name}.{sub.name}", "lineno": sub.lineno}))
        else:
            module_level.append(ast.get_source_segment(src, node) or "")
    if module_level:
        chunks.insert(0, Chunk("\n".join(module_level), {"method": "python_ast", "type": "module_level"}))
    return chunks

c_code = python_ast_chunk(SAMPLE_CODE, max_chars=300)
for c in c_code:
    print(f"[{c.metadata['type']}:{c.metadata.get('name', '-')}] {len(c.text)} ký tự")

# --- Bản LangChain (tách theo cú pháp ngôn ngữ, hỗ trợ nhiều ngôn ngữ) ---
try:
    from langchain_text_splitters import Language
    lc_code = RecursiveCharacterTextSplitter.from_language(Language.PYTHON, chunk_size=250, chunk_overlap=0)
    print("\n[LangChain] code splitter:", len(lc_code.split_text(SAMPLE_CODE)), "chunks")
except Exception as e:
    print("Bỏ qua LangChain:", e)

## B5. JSON chunking (giữ đường dẫn key)

In [ ]:
SAMPLE_JSON = {
    "cua_hang": "Nhà sách Cần Thơ",
    "sach": [
        {"id": 1, "ten": "Lập trình Python", "tac_gia": "Nguyễn Văn A", "gia": 120000, "tags": ["python", "lap-trinh"]},
        {"id": 2, "ten": "Học máy cơ bản", "tac_gia": "Trần Thị B", "gia": 150000, "tags": ["ml", "ai"]},
        {"id": 3, "ten": "Cơ sở dữ liệu", "tac_gia": "Lê Văn C", "gia": 99000, "tags": ["sql"]},
    ],
    "chi_nhanh": {"ninh_kieu": {"dia_chi": "1 Hòa Bình", "gio": "8-21h"}, "cai_rang": {"dia_chi": "2 Lê Bình", "gio": "7-20h"}},
}

def json_chunk(data, max_chars: int = 220) -> List[Chunk]:
    out: List[Chunk] = []
    dump = lambda o: json.dumps(o, ensure_ascii=False)

    def emit(obj, path):
        out.append(Chunk(dump(obj), {"method": "json", "path": path or "$"}))

    def rec(obj, path):
        if len(dump(obj)) <= max_chars or not isinstance(obj, (dict, list)):
            emit(obj, path); return
        if isinstance(obj, dict):
            buf: Dict[str, Any] = {}
            for k, v in obj.items():
                if len(dump(v)) > max_chars:
                    if buf: emit(buf, path); buf = {}
                    rec(v, f"{path}.{k}" if path else k)
                else:
                    if buf and len(dump({**buf, k: v})) > max_chars:
                        emit(buf, path); buf = {}
                    buf[k] = v
            if buf: emit(buf, path)
        else:
            buf_l: list = []
            for i, v in enumerate(obj):
                if len(dump(v)) > max_chars:
                    if buf_l: emit(buf_l, path); buf_l = []
                    rec(v, f"{path}[{i}]")
                else:
                    if buf_l and len(dump(buf_l + [v])) > max_chars:
                        emit(buf_l, path); buf_l = []
                    buf_l.append(v)
            if buf_l: emit(buf_l, path)

    rec(data, "")
    return out

c_json = json_chunk(SAMPLE_JSON, 220)
for c in c_json:
    print(f"{c.metadata['path']:14} {c.text[:110]}")

try:
    from langchain_text_splitters import RecursiveJsonSplitter
    print("\n[LangChain] RecursiveJsonSplitter:", len(RecursiveJsonSplitter(max_chunk_size=220).split_json(SAMPLE_JSON)), "chunks")
except Exception as e:
    print("Bỏ qua LangChain:", e)

## B6. Loader PDF / DOCX -> Markdown -> chunk theo cấu trúc
Hai hàm dưới biến file thật thành Markdown để dùng lại `element_chunk` / `markdown_header_chunk` ở trên. Cell cuối cho phép upload file trên Colab (bỏ qua nếu không upload).

In [ ]:
def docx_to_markdown(path: str) -> str:
    """DOCX -> Markdown: style 'Heading N' -> #, bảng -> bảng Markdown, danh sách -> '- '."""
    import docx
    from docx.table import Table
    from docx.text.paragraph import Paragraph
    d, out = docx.Document(path), []
    for child in d.element.body.iterchildren():
        if child.tag.endswith("}p"):
            p = Paragraph(child, d)
            if not p.text.strip():
                continue
            sty = (p.style.name or "").lower()
            if sty.startswith("heading"):
                m = re.search(r"(\d+)", sty)
                out.append("#" * int(m.group(1) if m else 1) + " " + p.text.strip())
            elif "list" in sty:
                out.append("- " + p.text.strip())
            else:
                out.append(p.text.strip())
        elif child.tag.endswith("}tbl"):
            t = Table(child, d)
            rows = [[c.text.strip().replace("\n", " ") for c in r.cells] for r in t.rows]
            if rows:
                out.append("| " + " | ".join(rows[0]) + " |")
                out.append("|" + "---|" * len(rows[0]))
                out += ["| " + " | ".join(r) + " |" for r in rows[1:]]
        out.append("")
    return "\n".join(out)

def pdf_to_markdown(path: str) -> str:
    """PDF (có text layer) -> Markdown. PDF scan cần OCR trước (vd. ocrmypdf)."""
    try:
        import pymupdf4llm
    except ImportError:
        os.system("pip install -q pymupdf4llm")
        import pymupdf4llm
    return pymupdf4llm.to_markdown(path)

# Thử nghiệm DOCX: tạo file mẫu rồi đọc lại
import docx as _docx
_d = _docx.Document()
_d.add_heading("Quy chế đào tạo", 1)
_d.add_heading("Điều 1. Phạm vi", 2)
_d.add_paragraph("Quy chế áp dụng cho toàn bộ sinh viên đại học chính quy.")
_d.add_heading("Điều 2. Học phí", 2)
_t = _d.add_table(rows=2, cols=2); _t.rows[0].cells[0].text = "Hệ"; _t.rows[0].cells[1].text = "Mức phí"
_t.rows[1].cells[0].text = "Chính quy"; _t.rows[1].cells[1].text = "10 triệu/năm"
_d.save("mau.docx")
md_from_docx = docx_to_markdown("mau.docx")
print(md_from_docx)
show(markdown_header_chunk(md_from_docx, max_level=2), n=5)

In [ ]:
# (Tuỳ chọn) Upload file PDF/DOCX của bạn trên Colab:
try:
    from google.colab import files
    up = files.upload()
    for name in up:
        md = pdf_to_markdown(name) if name.lower().endswith(".pdf") else docx_to_markdown(name)
        print(f"== {name}: {len(md)} ký tự ==")
        show(element_chunk(md, 800), n=5)
except Exception as e:
    print("Bỏ qua upload (không chạy trên Colab hoặc chưa chọn file):", type(e).__name__)

---
# C. Semantic chunking
Ý tưởng: tính embedding từng câu, **cắt ở chỗ nội dung thay đổi** (khoảng cách cosine giữa hai câu liền kề tăng đột biến).
## C1. Embedding breakpoints (4 cách chọn ngưỡng)
- `percentile`: cắt tại các khoảng cách lớn hơn phân vị thứ `threshold` (vd. 90).
- `stddev`: ngưỡng = mean + `threshold` x std.
- `iqr`: ngưỡng = Q3 + `threshold` x IQR (outlier theo IQR).
- `gradient`: dùng đạo hàm của chuỗi khoảng cách (hợp với văn bản có nhiều chủ đề gần nhau).
- `buffer_size`: gộp ± k câu lân cận khi embedding để giảm nhiễu. Sau đó ép `min_chars`/`max_chars`.

In [ ]:
def semantic_breakpoints(sents: List[str], buffer_size: int = 1, method: str = "percentile",
                         threshold: float = 90):
    combined = [" ".join(sents[max(0, i - buffer_size): i + buffer_size + 1]) for i in range(len(sents))]
    E = embed(combined)
    dist = 1.0 - np.sum(E[:-1] * E[1:], axis=1)                   # khoảng cách cosine câu i -> i+1
    if method == "percentile":
        score, th = dist, np.percentile(dist, threshold)
    elif method == "stddev":
        score, th = dist, dist.mean() + threshold * dist.std()
    elif method == "iqr":
        q1, q3 = np.percentile(dist, [25, 75])
        score, th = dist, q3 + threshold * (q3 - q1)
    elif method == "gradient":
        score = np.gradient(dist)
        th = np.percentile(score, threshold)
    else:
        raise ValueError(method)
    breaks = np.where(score > th)[0]                              # cắt SAU câu i
    return breaks, dist, score, th

def semantic_chunk(text: str, buffer_size: int = 1, method: str = "percentile",
                   threshold: float = 90, min_chars: int = 150, max_chars: int = 900,
                   return_debug: bool = False):
    sents = split_sentences(text)
    if len(sents) < 3:
        return (sents, None) if return_debug else sents
    breaks, dist, score, th = semantic_breakpoints(sents, buffer_size, method, threshold)
    groups, start = [], 0
    for b in breaks:
        groups.append(sents[start:b + 1]); start = b + 1
    groups.append(sents[start:])
    chunks = [" ".join(g) for g in groups if g]

    merged = []                                                   # gộp chunk quá ngắn vào chunk trước
    for c in chunks:
        if merged and len(merged[-1]) < min_chars:
            merged[-1] += " " + c
        else:
            merged.append(c)
    if len(merged) > 1 and len(merged[-1]) < min_chars:
        merged[-2] += " " + merged.pop()
    final = []                                                    # tách chunk quá dài theo câu
    for c in merged:
        final.extend([c] if len(c) <= max_chars else sentence_chunk(c, max_chars, 0))
    if return_debug:
        return final, {"sents": sents, "dist": dist, "score": score, "threshold": th, "breaks": breaks}
    return final

c_sem, dbg = semantic_chunk(DOC_PLAIN, buffer_size=1, method="percentile", threshold=85,
                            min_chars=150, max_chars=900, return_debug=True)
show(c_sem, n=10, width=170)

# Trực quan hoá: đỉnh vượt ngưỡng = ranh giới chunk
plt.figure(figsize=(11, 3.6))
plt.plot(dbg["score"], marker="o", ms=3, label="khoảng cách giữa câu i và i+1")
plt.axhline(dbg["threshold"], color="red", ls="--", label="ngưỡng")
for b in dbg["breaks"]:
    plt.axvline(b, color="orange", alpha=0.3)
plt.xlabel("chỉ số câu"); plt.ylabel("cosine distance"); plt.legend(); plt.title("Semantic breakpoints")
plt.tight_layout(); plt.show()

# So sánh các phương pháp ngưỡng
for m, th in [("percentile", 85), ("stddev", 1.0), ("iqr", 1.0), ("gradient", 85)]:
    r = semantic_chunk(DOC_PLAIN, 1, m, th, return_debug=False)
    print(f"{m:10} threshold={th:<5} -> {len(r)} chunks")

## C2. Lexical TextTiling (không cần embedding / không cần mô hình)
Thuật toán Hearst: so sánh từ vựng (TF-IDF) giữa hai khối câu liền kề, tính **depth score** của các "thung lũng" tương đồng -> cắt ở thung lũng sâu. Rất nhanh, chạy offline, độc lập ngôn ngữ.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def _depth_scores(s: np.ndarray) -> np.ndarray:
    d = np.zeros(len(s))
    for i in range(len(s)):
        l = i
        while l > 0 and s[l - 1] >= s[l]: l -= 1
        r = i
        while r < len(s) - 1 and s[r + 1] >= s[r]: r += 1
        d[i] = (s[l] - s[i]) + (s[r] - s[i])
    return d

def texttiling_chunk(text: str, block: int = 3, smooth: int = 1, k: float = 0.5,
                     min_sents: int = 2) -> List[str]:
    sents = split_sentences(text)
    if len(sents) < 2 * block:
        return [" ".join(sents)]
    X = TfidfVectorizer(sublinear_tf=True, token_pattern=r"(?u)\b\w+\b").fit_transform(sents).toarray()
    sims = []
    for g in range(1, len(sents)):                               # khe giữa câu g-1 và g
        a = X[max(0, g - block):g].sum(0); b = X[g:g + block].sum(0)
        den = np.linalg.norm(a) * np.linalg.norm(b)
        sims.append(float(a @ b / den) if den else 0.0)
    sims = np.array(sims)
    if smooth:
        sims = np.convolve(sims, np.ones(2 * smooth + 1) / (2 * smooth + 1), mode="same")
    depth = _depth_scores(sims)
    cutoff = depth.mean() + k * depth.std()
    cuts = [i + 1 for i in range(len(depth)) if depth[i] > cutoff]     # cắt trước câu i+1
    chunks, start = [], 0
    for c in cuts:
        if c - start >= min_sents:
            chunks.append(" ".join(sents[start:c])); start = c
    chunks.append(" ".join(sents[start:]))
    return chunks

c_tt = texttiling_chunk(DOC_PLAIN, block=3, smooth=1, k=0.5)
show(c_tt, n=8, width=150)

## C3. Clustering liền kề (Agglomerative có ràng buộc chuỗi)
Gom các câu thành cụm theo embedding nhưng chỉ cho phép gộp **các câu kề nhau** (connectivity = chuỗi) nên kết quả luôn là các đoạn liên tục. Điều khiển bằng `n_chunks` hoặc `distance_threshold`.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.sparse import diags

def cluster_chunk(text: str, n_chunks: Optional[int] = None, distance_threshold: float = 0.55) -> List[str]:
    sents = split_sentences(text)
    n = len(sents)
    if n < 3:
        return sents
    E = embed(sents)
    conn = diags([1, 1], [-1, 1], shape=(n, n)).tocsr()
    model = AgglomerativeClustering(
        n_clusters=n_chunks, distance_threshold=None if n_chunks else distance_threshold,
        metric="cosine", linkage="average", connectivity=conn)
    labels = model.fit_predict(E)
    chunks, cur = [], [sents[0]]
    for i in range(1, n):
        if labels[i] != labels[i - 1]:
            chunks.append(" ".join(cur)); cur = []
        cur.append(sents[i])
    chunks.append(" ".join(cur))
    return chunks

c_clu = cluster_chunk(DOC_PLAIN, n_chunks=7)
show(c_clu, n=8, width=150)

## C4. LangChain `SemanticChunker` (so sánh với bản tự cài đặt)

In [ ]:
try:
    from langchain_experimental.text_splitter import SemanticChunker
    from langchain_huggingface import HuggingFaceEmbeddings
    lc_emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
    lc_sem = SemanticChunker(lc_emb, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=85)
    c_lc_sem = [d.page_content for d in lc_sem.create_documents([DOC_PLAIN])]
    show(c_lc_sem, n=6, width=150)
except Exception as e:
    c_lc_sem = None
    print("Bỏ qua LangChain SemanticChunker:", type(e).__name__, e)

### Kết hợp tốt nhất trong thực tế: Structure-based -> Semantic
Tách theo tiêu đề trước (giữ cấu trúc), rồi chỉ chạy semantic trong những section còn dài.

In [ ]:
def hybrid_chunk(md: str, max_section_chars: int = 700, **sem_kw) -> List[Chunk]:
    out = []
    for sec in markdown_header_chunk(md, max_level=3):
        plain = md_to_plain(sec.text)
        if len(plain) <= max_section_chars:
            out.append(Chunk(f"[{sec.metadata['header_path']}]\n{plain}", sec.metadata))
        else:
            for j, piece in enumerate(semantic_chunk(plain, **sem_kw)):
                out.append(Chunk(f"[{sec.metadata['header_path']}]\n{piece}", {**sec.metadata, "sub_idx": j}))
    return out

c_hybrid = hybrid_chunk(DOC_MD, max_section_chars=600, buffer_size=1, method="percentile", threshold=80,
                        min_chars=120, max_chars=700)
show(c_hybrid, n=5, width=200)

---
# D. Kỹ thuật nâng cao
## D1. Parent-Child (small-to-big / auto-merging)
Embedding **chunk con (nhỏ, chính xác)** để tìm kiếm, nhưng trả về **chunk cha (lớn, đủ ngữ cảnh)** cho LLM.

In [ ]:
def parent_child_chunk(text: str, parent_size: int = 1000, child_size: int = 250, child_overlap: int = 30):
    parents = recursive_chunk(text, parent_size, 0)
    parent_store, children = {}, []
    for pi, p in enumerate(parents):
        pid = f"p{pi}"
        parent_store[pid] = p
        for ci, c in enumerate(recursive_chunk(p, child_size, child_overlap)):
            children.append(Chunk(c, {"method": "parent_child", "parent_id": pid, "child_idx": ci}))
    return parent_store, children

parent_store, children = parent_child_chunk(DOC_PLAIN, 1000, 250, 30)
print(f"{len(parent_store)} parent, {len(children)} child")

child_emb = embed([c.text for c in children])
def retrieve_parent(query: str, k: int = 2):
    q = embed([query])[0]
    sims = child_emb @ q
    seen, res = set(), []
    for i in np.argsort(-sims):
        pid = children[i].metadata["parent_id"]
        if pid not in seen:
            seen.add(pid); res.append((float(sims[i]), children[i].text, parent_store[pid]))
        if len(res) == k: break
    return res

for score, child, parent in retrieve_parent("Lẩu mắm nấu từ nguyên liệu gì?", 1):
    print(f"\nscore={score:.3f}\nCHILD  : {child}\nPARENT : {parent[:400]}...")

## D2. Late chunking
Thay vì cắt rồi mới embedding (mỗi chunk mất ngữ cảnh của cả văn bản), ta **đưa toàn bộ văn bản qua mô hình trước**, sau đó mean-pool các vector token **theo từng chunk**. Mỗi chunk vì vậy "nhớ" ngữ cảnh toàn văn (ví dụ giải quyết đại từ "thành phố này"). Cần mô hình context dài (vd. `jinaai/jina-embeddings-v3`, `BAAI/bge-m3` 8192 token); demo dưới dùng mô hình hiện tại (512 token) với văn bản ngắn.

In [ ]:
import torch

def chunk_spans(text: str, chunks: List[str]) -> List[Tuple[int, int]]:
    spans, pos = [], 0
    for c in chunks:
        s = text.find(c, pos)
        if s == -1:
            raise ValueError("Chunk phải là chuỗi con nguyên văn của text (không chuẩn hoá/overlap)")
        spans.append((s, s + len(c))); pos = s + len(c)
    return spans

@torch.no_grad()
def late_chunk_embeddings(text: str, spans: List[Tuple[int, int]], max_len: int = 512) -> np.ndarray:
    hf_model = embedder[0].auto_model
    enc = hf_tokenizer(text, return_offsets_mapping=True, return_tensors="pt",
                       truncation=True, max_length=max_len)
    offsets = enc.pop("offset_mapping")[0].tolist()
    hidden = hf_model(**{k: v.to(hf_model.device) for k, v in enc.items()}).last_hidden_state[0]
    embs = []
    for cs, ce in spans:
        idx = [i for i, (a, b) in enumerate(offsets) if b > a and a >= cs and b <= ce]
        if not idx:
            embs.append(torch.zeros(hidden.shape[-1])); continue
        embs.append(hidden[idx].mean(0).cpu())
    M = torch.stack(embs)
    return torch.nn.functional.normalize(M, dim=-1).numpy()

LATE_TEXT = ("Cần Thơ là thành phố lớn nhất vùng Đồng bằng sông Cửu Long. "
             "Thành phố này nằm bên bờ phải sông Hậu. "
             "Nơi đây nổi tiếng với chợ nổi Cái Răng họp từ sáng sớm.")
late_chunks = [s for s in split_sentences(LATE_TEXT)]
spans = chunk_spans(LATE_TEXT, late_chunks)

E_late = late_chunk_embeddings(LATE_TEXT, spans)
E_naive = embed(late_chunks)
q = embed(["Cần Thơ nằm bên bờ sông nào?"])[0]
print("Chunk 2 = 'Thành phố này nằm bên bờ phải sông Hậu.' (đại từ 'thành phố này' mất ngữ cảnh khi cắt rời)")
print(f"  cosine(query, chunk) - naive: {E_naive[1] @ q:.3f} | late chunking: {E_late[1] @ q:.3f}")

## D3. LLM / Agentic chunking + Propositional + Contextual chunking
Cần một hàm `llm(prompt) -> str`. Chọn một backend (Anthropic / OpenAI / Gemini) hoặc dùng `mock_llm` để chạy thử pipeline offline.

- **LLM chunking:** đánh số câu, nhờ LLM nhóm các câu liền kề theo chủ đề (rẻ hơn nhiều so với bắt LLM chép lại văn bản).
- **Propositional chunking:** LLM tách văn bản thành các mệnh đề độc lập, tự đầy đủ nghĩa.
- **Contextual chunking (Anthropic):** LLM viết 1-2 câu ngữ cảnh đặt trước mỗi chunk trước khi embedding.

In [ ]:
# ---- Chọn backend LLM (điền API key vào Colab Secrets hoặc biến môi trường) ----
# Cài thêm khi dùng LLM thật:  !pip install -q anthropic   (hoặc openai)
def _get_key(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def make_anthropic_llm(model: str = "claude-haiku-4-5", max_tokens: int = 2000) -> Callable[[str], str]:
    import anthropic
    client = anthropic.Anthropic(api_key=_get_key("ANTHROPIC_API_KEY"))
    def llm(prompt: str) -> str:
        r = client.messages.create(model=model, max_tokens=max_tokens,
                                   messages=[{"role": "user", "content": prompt}])
        return r.content[0].text
    return llm

def make_openai_llm(model: str = "gpt-4o-mini") -> Callable[[str], str]:
    from openai import OpenAI
    client = OpenAI(api_key=_get_key("OPENAI_API_KEY"))
    def llm(prompt: str) -> str:
        r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return llm

def mock_llm(prompt: str) -> str:
    """LLM giả lập để chạy thử offline: nhóm mỗi 4 câu một chunk / tách mệnh đề theo dấu phẩy."""
    if '"chunks"' in prompt:
        n = len(re.findall(r"^\[\d+\]", prompt, flags=re.M))
        return json.dumps({"chunks": [list(range(i, min(i + 4, n))) for i in range(0, n, 4)]})
    if '"propositions"' in prompt:
        body = prompt.split("VĂN BẢN:")[-1]
        return json.dumps({"propositions": [p.strip() for p in re.split(r"[.;]", body) if len(p.strip()) > 15]})
    return "Đoạn này thuộc tài liệu tổng hợp về RAG và thành phố Cần Thơ."

# Chọn backend: nếu có key thì dùng LLM thật, nếu không dùng mock
llm = mock_llm
try:
    if _get_key("ANTHROPIC_API_KEY"):
        llm = make_anthropic_llm()
    elif _get_key("OPENAI_API_KEY"):
        llm = make_openai_llm()
except Exception as e:
    print("Không khởi tạo được LLM thật, dùng mock:", e)
print("LLM đang dùng:", "MOCK (kết quả chỉ mang tính minh hoạ)" if llm is mock_llm else "LLM thật")

In [ ]:
def _parse_json(s: str):
    m = re.search(r"\{.*\}", s, flags=re.S)
    return json.loads(m.group(0)) if m else None

def llm_chunk(text: str, llm: Callable[[str], str], max_sents_per_call: int = 80) -> List[str]:
    """LLM nhóm các câu liên tiếp theo chủ đề. Có kiểm tra tính hợp lệ + fallback."""
    sents = split_sentences(text)
    chunks: List[str] = []
    for off in range(0, len(sents), max_sents_per_call):
        part = sents[off: off + max_sents_per_call]
        numbered = "\n".join(f"[{i}] {s}" for i, s in enumerate(part))
        prompt = f"""Bạn là chuyên gia chuẩn bị dữ liệu cho hệ thống RAG.
Dưới đây là các câu được đánh số. Hãy nhóm các câu LIÊN TIẾP thành các đoạn (chunk) sao cho mỗi đoạn
nói về MỘT chủ đề/ý hoàn chỉnh, độ dài mỗi đoạn khoảng 3-8 câu. Mỗi câu thuộc đúng một đoạn, giữ nguyên thứ tự.
Chỉ trả về JSON dạng {{"chunks": [[0,1,2],[3,4],...]}}, không giải thích.

{numbered}"""
        groups = None
        try:
            groups = (_parse_json(llm(prompt)) or {}).get("chunks")
        except Exception as e:
            print("LLM lỗi, dùng fallback:", e)
        flat = [i for g in (groups or []) for i in g]
        if flat != list(range(len(part))):                    # không hợp lệ -> fallback theo câu
            groups = [list(range(len(part)))]
            print("Kết quả LLM không hợp lệ -> dùng một chunk duy nhất cho phần này")
        chunks += [" ".join(part[i] for i in g) for g in groups]
    return chunks

def propositional_chunk(text: str, llm: Callable[[str], str]) -> List[str]:
    prompt = f"""Hãy tách văn bản thành các MỆNH ĐỀ (proposition): mỗi mệnh đề là một câu ngắn, độc lập, nêu một sự thật duy nhất,
thay đại từ bằng danh từ cụ thể để hiểu được mà không cần ngữ cảnh. Chỉ trả về JSON {{"propositions": ["...", "..."]}}.

VĂN BẢN:
{text}"""
    try:
        return (_parse_json(llm(prompt)) or {}).get("propositions", [])
    except Exception as e:
        print("LLM lỗi:", e); return []

def contextualize_chunks(doc: str, chunks: List[str], llm: Callable[[str], str]) -> List[str]:
    """Contextual Retrieval: thêm 1-2 câu ngữ cảnh (đặt chunk vào toàn văn) trước mỗi chunk."""
    out = []
    for c in chunks:
        prompt = f"""<document>\n{doc[:12000]}\n</document>
Đây là một đoạn trích từ tài liệu trên:
<chunk>\n{c}\n</chunk>
Hãy viết 1-2 câu ngắn đặt đoạn này vào ngữ cảnh toàn tài liệu để hỗ trợ truy xuất. Chỉ trả về phần ngữ cảnh."""
        out.append(llm(prompt).strip() + "\n" + c)
    return out

c_llm = llm_chunk(DOC_PLAIN, llm)
print("LLM chunking:"); show(c_llm, n=4, width=150)

c_prop = propositional_chunk(" ".join(split_sentences(DOC_PLAIN)[-6:]), llm)
print("\nPropositional:"); show(c_prop, n=4, width=150)

c_ctx = contextualize_chunks(DOC_PLAIN, c_sent[:2], llm)
print("\nContextual:"); show(c_ctx, n=2, width=220)

---
# E. Đánh giá & so sánh các chiến lược
1. **Thống kê**: số chunk, độ dài, phần trăm chunk kết thúc đúng cuối câu (chất lượng ranh giới).
2. **Truy xuất**: với bộ câu hỏi mẫu, chunk nào chứa đáp án nằm trong top-k? (Hit@k, MRR). Hãy thay bằng bộ câu hỏi thật của bạn để chọn chiến lược cho dữ liệu của mình.

In [ ]:
STRATEGIES: Dict[str, List[str]] = {
    "fixed_400":        c_fixed,
    "token_100":        c_tok_hf,
    "sentence_450":     c_sent,
    "paragraph_700":    c_para,
    "recursive_450":    c_rec,
    "md_header":        texts_of(c_md_split),
    "element_aware":    texts_of(c_elem),
    "semantic_pct85":   c_sem,
    "texttiling":       c_tt,
    "cluster_7":        c_clu,
    "hybrid":           texts_of(c_hybrid),
}
if c_lc_sem:
    STRATEGIES["langchain_semantic"] = c_lc_sem

END_OK = re.compile(r"[.!?…:;)\]\"”`]$")
rows = []
for name, chs in STRATEGIES.items():
    L = [len(c) for c in chs]
    rows.append({"strategy": name, "n_chunks": len(chs), "mean_chars": round(np.mean(L)),
                 "min": min(L), "max": max(L),
                 "mean_tokens": round(np.mean([count_tokens(c) for c in chs])),
                 "end_at_sentence_%": round(100 * np.mean([bool(END_OK.search(c.strip())) for c in chs]))})
df_stats = pd.DataFrame(rows).set_index("strategy")
df_stats

In [ ]:
QA = [  # (câu hỏi, từ khoá bắt buộc phải xuất hiện trong chunk chứa đáp án)
    ("Chunk quá lớn gây ra vấn đề gì?",               ["loãng"]),
    ("Nên dùng overlap khoảng bao nhiêu phần trăm?",   ["10 đến 20 phần trăm"]),
    ("Chợ nổi Cái Răng họp vào lúc nào?",              ["sáng sớm"]),
    ("Lẩu mắm được nấu từ loại mắm nào?",              ["mắm cá linh"]),
    ("Những cơ sở dữ liệu vector nào được nhắc đến?",  ["FAISS"]),
    ("Cần Thơ nằm bên bờ sông nào?",                   ["sông Hậu"]),
    ("Ưu điểm của semantic chunking là gì?",           ["mạch lạc theo chủ đề"]),
    ("Biệt danh của Cần Thơ là gì?",                   ["Tây Đô"]),
]

def evaluate_retrieval(chunks: List[str], k: int = 3):
    E = embed(chunks)
    hits, rr = 0, 0.0
    for q, kws in QA:
        sims = E @ embed([q])[0]
        order = np.argsort(-sims)[:k]
        rank = next((r for r, i in enumerate(order, 1) if all(kw.lower() in chunks[i].lower() for kw in kws)), None)
        if rank:
            hits += 1; rr += 1 / rank
    return hits / len(QA), rr / len(QA)

res = []
for name, chs in STRATEGIES.items():
    h1, m1 = evaluate_retrieval(chs, 1)
    h3, m3 = evaluate_retrieval(chs, 3)
    res.append({"strategy": name, "Hit@1": round(h1, 2), "Hit@3": round(h3, 2), "MRR@3": round(m3, 2)})
df_ret = pd.DataFrame(res).set_index("strategy").sort_values(["Hit@1", "MRR@3"], ascending=False)
df_ret

In [ ]:
# Kết hợp 2 bảng và vẽ biểu đồ
df_all = df_stats.join(df_ret)
ax = df_all[["Hit@1", "Hit@3"]].sort_values("Hit@3").plot.barh(figsize=(8, 4.5), width=0.75)
ax.set_xlim(0, 1.05); ax.set_title("Hit@k trên bộ câu hỏi mẫu (càng cao càng tốt)"); plt.tight_layout(); plt.show()

## Xuất chunk ra file (JSONL) để nạp vào vector DB

In [ ]:
def export_jsonl(chunks, path: str, source: str = "doc_mau"):
    with open(path, "w", encoding="utf-8") as f:
        for i, c in enumerate(chunks):
            text = c.text if isinstance(c, Chunk) else c
            meta = c.metadata if isinstance(c, Chunk) else {}
            f.write(json.dumps({"id": f"{source}-{i}", "text": text, "n_tokens": count_tokens(text),
                                "source": source, **{k: v for k, v in meta.items() if k != "idx"}},
                               ensure_ascii=False) + "\n")
    print(f"Đã ghi {len(chunks)} chunk -> {path}")

export_jsonl(c_elem, "chunks_element.jsonl")
# Gợi ý nạp vào vector DB (ví dụ Chroma):
#   !pip install -q chromadb
#   import chromadb; col = chromadb.Client().create_collection("demo")
#   col.add(ids=[...], documents=[...], embeddings=embed([...]).tolist(), metadatas=[...])

---
# Hướng dẫn chọn chiến lược & tham số

| Loại dữ liệu / mục tiêu | Chiến lược gợi ý | Tham số khởi điểm |
|---|---|---|
| Văn bản thuần, cần nhanh, làm baseline | **Recursive** | 400-512 token, overlap 10-15% |
| Tài liệu có tiêu đề (Markdown, HTML, Word, PDF đã chuyển MD) | **Markdown/Element-aware** -> recursive cho section dài | max 500-800 token, luôn gắn `header_path` |
| Bảng, code, JSON | Chunk theo cấu trúc (không cắt giữa bảng/hàm), lặp tiêu đề cột/khai báo lớp | - |
| Văn bản dài, nhiều chủ đề, không có cấu trúc (transcript, bài báo) | **Semantic** (percentile 85-95) hoặc **TextTiling** nếu cần rẻ/offline | min 150 / max 900 ký tự, `buffer_size=1` |
| Cần độ chính xác truy xuất cao + ngữ cảnh rộng cho LLM | **Parent-Child** hoặc **Sentence-window** | child 200-300, parent 800-1500 token |
| Chunk hay chứa đại từ/tham chiếu mơ hồ | **Contextual chunking** hoặc **Late chunking** | cần LLM / mô hình long-context |
| Chất lượng là ưu tiên, ngân sách thoải mái | **LLM/Agentic**, Propositional | cache kết quả, kiểm tra JSON |
| Production | **Hybrid:** structure -> semantic/recursive -> thêm metadata | đánh giá bằng bộ câu hỏi thật |

**Lưu ý quan trọng**
- Kích thước chunk phải **≤ giới hạn token của mô hình embedding** (MiniLM: 128-256, e5/bge-m3: 512-8192).
- Không có chiến lược "tốt nhất cho mọi trường hợp": hãy dùng mục E với **bộ câu hỏi của chính bạn** để chọn.
- Tiếng Việt: tách câu dựa trên dấu câu hoạt động tốt; nếu cần chính xác hơn có thể dùng `underthesea` (`sent_tokenize`) để thay `split_sentences`.
- Luôn lưu metadata (nguồn, trang, `header_path`, vị trí) để trích dẫn và lọc khi truy xuất.